In [1]:
# ============================================
# TRAIN MODELS DIRECTLY ON REAL DATA (80/20)
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import recall_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
import joblib

# -----------------------------
# Load real cleaned dataset
# -----------------------------
import os
data_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'data', 'synthetic_maternal_cleaned.csv')
df = pd.read_csv(data_path)

X = df.drop("RiskLevel", axis=1)
y = df["RiskLevel"]

# -----------------------------
# Encode labels
# -----------------------------
le = LabelEncoder()
y_enc = le.fit_transform(y)

# -----------------------------
# 80/20 Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# -----------------------------
# Define models
# -----------------------------
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=5000))
    ]),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        num_class=len(np.unique(y_enc)),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42
    )
}

# Voting & Stacking Ensembles
estimators = [
    ("lr", LogisticRegression(max_iter=5000)),
    ("dt", DecisionTreeClassifier()),
    ("rf", RandomForestClassifier()),
    ("gb", GradientBoostingClassifier())
]

models["Voting Ensemble"] = VotingClassifier(
    estimators=estimators,
    voting="soft"
)

models["Stacking Ensemble"] = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=5000)
)

# -----------------------------
# Train + Evaluate + Save Models
# -----------------------------
results = {}

import os
os.makedirs("../models", exist_ok=True)


for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    macro_recall = recall_score(y_test, preds, average="macro")
    results[name] = macro_recall
    
    print(f"Macro Recall: {macro_recall:.4f}")
    print(classification_report(y_test, preds))
    
    # Save model
    joblib.dump(model, f"../models/{name.replace(' ', '_')}.pkl")

# -----------------------------
# Comparison Table
# -----------------------------
comparison_df = pd.DataFrame.from_dict(results, orient="index", columns=["Macro Recall"])
comparison_df = comparison_df.sort_values(by="Macro Recall", ascending=False)

comparison_df



Training Logistic Regression...
Macro Recall: 0.4625
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         5
           1       0.82      0.93      0.87       227
           2       0.66      0.45      0.54        86

    accuracy                           0.79       318
   macro avg       0.49      0.46      0.47       318
weighted avg       0.77      0.79      0.77       318


Training Decision Tree...
Macro Recall: 0.9197
              precision    recall  f1-score   support

           0       0.80      0.80      0.80         5
           1       0.99      0.98      0.99       227
           2       0.95      0.98      0.97        86

    accuracy                           0.98       318
   macro avg       0.92      0.92      0.92       318
weighted avg       0.98      0.98      0.98       318


Training Random Forest...
Macro Recall: 0.9202
              precision    recall  f1-score   support

           0       1.00      0.80

,Macro Recall
Gradient Boosting,0.922645
XGBoost,0.922645
Voting Ensemble,0.922645
Random Forest,0.920237
Decision Tree,0.919708
Stacking Ensemble,0.915832
Logistic Regression,0.462470
